In [11]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import json
from datasets import load_dataset, DatasetDict
import chromadb
from rag_bench import evaluator, helper

from pipeline.ingestion.chroma import insert_data

### Загрузка датасетов

In [6]:
CACHE_DIR = "../datasets/"

HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"

PUBLIC_TEXTS_REPO: str = "ai-forever/hist-rag-bench-public-texts"
PUBLIC_QA_REPO: str = "ai-forever/hist-rag-bench-public-questions"


def get_private_qa_dataset(version):
    return load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version, cache_dir=CACHE_DIR)


def get_private_texts_dataset(version):
    return load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version, cache_dir=CACHE_DIR)

# map public_id:private_id
def get_public_to_private_texts_mapping(version):
    private_texts_ds = get_private_texts_dataset(version)
    mapping = {}
    for item in private_texts_ds["train"]:
        mapping[item["public_id"]] = item["id"]
    return mapping


version = helper.get_latest_version(helper.get_ds_versions(PUBLIC_TEXTS_REPO))
texts_ds = load_dataset(PUBLIC_TEXTS_REPO, revision=version, cache_dir=CACHE_DIR)
questions_ds = load_dataset(PUBLIC_QA_REPO, revision=version, cache_dir=CACHE_DIR)
private_texts_ds = get_private_texts_dataset(version)
qa_dataset = get_private_qa_dataset(version)
mapping = get_public_to_private_texts_mapping(version)
version

'1.15.0'

### Создание базы

In [7]:
COLLECTION_NAME: str = "dragon"
CHROMA_PATH: str = "../chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

def create_collection(chroma_client, name):
    collection = None
    try:
        chroma_client.delete_collection(name)
    except Exception as e:
        print(f"Коллекция {name} уже существует и будет перезатерта: {e}")
    finally:
        collection = chroma_client.create_collection(
            name=name,
            metadata={
                "hnsw:space": "cosine",
            }
        )
        print(f"Коллекция {name} создана по пути {CHROMA_PATH}")
    return collection


In [ ]:
chunk_size = 2000
chunk_overlap = 20
character_collection = create_collection(chroma_client=chroma_client, name="character")
insert_data(
    dataset=texts_ds,
    collection=character_collection,
    split_type="recursive_character",
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

### Оценка ретривера

In [ ]:
from pipeline.retriever import ChromaRetriever

def get_retrivier_result(query, collection):
    results = ChromaRetriever(collection).top_n(query=query, n=3)
    found_ids = [int(x["id"]) for x in results["metadatas"][0]]
    relevant_chunks = results['documents'][0]
    metadatas = results['metadatas'][0]
    return found_ids, relevant_chunks, metadatas

def create_context(q_id, relevant_chunks, metadatas, separator="\n\n---\n\n"):
    contexts_data = {}
    contexts_data[q_id] = {
        "relevant_chunks": relevant_chunks,
        "metadatas": metadatas
    }
    context_parts = []
    for i, (chunk, metadata) in enumerate(zip(relevant_chunks, metadatas)):
        score = metadata.get('score', 0)
        context_parts.append(f"[Документ {i+1} (релевантность: {score:.3f})]\n{chunk}")
    context = separator.join(context_parts)
    return f'Контекст: {context}'

results = {}
context = {}
public_questions = [{"id": str(row["public_id"]), "question": row["question"]} for row in qa_dataset["train"]]

for q in public_questions:
    q_id = q["id"]
    question = q["question"]
    found_ids, relevant_chunks, metadatas = get_retrivier_result(question, character_collection)
    context[q_id] = create_context(q_id, relevant_chunks, metadatas)
    results[q_id] = {
        "found_ids": found_ids,
        "model_answer": "-",
    }

ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())
ev_res["average_metrics"]["retrieval"]

### Оценка генератора

In [ ]:
from pipeline.generator import OpenRouterGenerator, GigaChatGenerator, LocalOllamaGenerator
from pipeline.retriever import ChromaRetriever

generator = LocalOllamaGenerator()
# generator = GigaChatGenerator()
# generator.model = "GigaChat-Max"
# generator = OpenRouterGenerator()

small_qa_ds = DatasetDict({
    "train": qa_dataset["train"].select(range(min(5, len(qa_dataset["train"]))))
})

def create_context(relevant_chunks, metadatas, separator="\n\n---\n\n"):
    context_parts = []
    for i, (chunk, metadata) in enumerate(zip(relevant_chunks, metadatas)):
        score = metadata.get('score', 0)
        context_parts.append(f"[Документ {i+1} (релевантность: {score:.3f})]\n{chunk}")
    context = separator.join(context_parts)
    return context


def get_generator_answers(query, collection):
    found_ids, relevant_chunks, metadatas = get_retrivier_result(query, collection)
    context = create_context(relevant_chunks, metadatas)
    answer = generator.generate(context=context, query=query)
    return found_ids, answer

retriever_result = results
results = {}
public_questions = [{"id": str(row["public_id"]), "question": row["question"]} for row in small_qa_ds["train"]]

for q in public_questions:
    q_id = q["id"]
    question = q["question"]
    found_ids, answer = get_generator_answers(question, character_collection)
    results[q_id] = {
        "found_ids": found_ids,
        "model_answer": answer,
    }

ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())
ev_res["average_metrics"]

In [ ]:
with open("../results/dragon_chroma_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ Результаты сохранены: results/dragon_chroma_results.json")